# ChronoSwan Intraday ES Impulse Chartbook

This notebook is the local, executed research chartbook for the intraday branch of ChronoSwan. It studies whether large 60-minute ES1 moves have recurring cross-asset drivers, whether simple conditional correlations already explain them, and whether PCA adds a useful factor-concentration view.

Raw Bloomberg workbooks, generated parquet files, figures, and executed reports stay local. The public repo carries methodology and sanitized examples only.


## Page 0 - Research Question And Contribution

**Question.** Is it useful to condition cross-asset PCA on large ES1 60-minute moves instead of fitting PCA on all intraday bars?

**Prior literature in brief.** Extreme correlations, contagion/interdependence, DCC, spillover networks, and PCA absorption ratios already cover adjacent territory. The contribution under test is narrower: a point-in-time ES impulse workflow that benchmarks conditional correlation before PCA.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from chronoswan.experiments.intraday_impulse_pca import (
    LITERATURE_CONTEXT,
    build_predictive_feature_frame,
    run_intraday_impulse_pca,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

plt.rcParams.update({
    "figure.figsize": (11, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

input_path = project_root / "data" / "17sheets.xlsx"
if not input_path.exists():
    input_path = project_root / "data" / "raw" / "17sheets.xlsx"

output_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)

result = run_intraday_impulse_pca(input_path=input_path, output_dir=output_dir)
long_frame = result["long_frame"]
coverage = result["coverage"]
price_panel = result["price_panel"]
return_panel = result["return_panel"]
events = result["events"]
event_summary = result["event_summary"]
corr = result["conditional_correlations"]
pca_summary = result["pca_summary"]
pca_loadings = result["pca_loadings"]
rolling_pca = result["rolling_pca"]
predictive = result["predictive_results"]
coefficients = result["predictive_coefficients"]
predictive_features = build_predictive_feature_frame(return_panel, price_panel, events)

print(f"Workbook: {input_path}")
print(f"Tickers: {coverage['ticker'].nunique()}")
print(f"Return grid: {return_panel.index.min()} to {return_panel.index.max()}")


The workbook parses cleanly into the ChronoSwan pipeline. This is a private local run; raw data and derived caches are intentionally excluded from git.


In [ ]:
literature = pd.DataFrame(LITERATURE_CONTEXT)[["topic", "anchor", "implication"]]
literature


The literature boundary is clear: PCA, crisis correlations, and factor concentration are established ideas. The contribution has to be the point-in-time ES impulse design and the comparison against conditional correlation.


## Page 1 - Raw Workbook Audit

Before any visualization or model, the manual Bloomberg workbook is normalized into one OHLCV table. Each sheet becomes timestamped rows tagged by ticker and asset class.


In [ ]:
long_frame.head(5)


This five-row preview is the normalized private OHLCV structure. It confirms that manual Bloomberg sheets are converted into a consistent research table before features are built.


In [ ]:
coverage.sort_values("rows", ascending=False).head(21)


Instrument clocks are uneven. Futures and FX have near-round-the-clock coverage, while ETFs and VIX-like series have cash-session coverage, so exact-overlap PCA samples are naturally smaller.


In [ ]:
plot_coverage = coverage.sort_values("rows", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(plot_coverage["ticker"].isin(["ES1", "NQ1", "RTY1"]), "#1f6f8b", "#7f8b99")
ax.barh(plot_coverage["ticker"], plot_coverage["rows"], color=colors)
ax.set_title("Intraday workbook coverage by ticker")
ax.set_xlabel("OHLCV rows")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_coverage_by_ticker.png", dpi=160)
plt.show()


Coverage is broad enough for a proof-of-concept cross-asset study. The shorter ETF and index clocks are a reason to report complete-row counts next to every PCA result.


In [ ]:
sample_return_columns = [c for c in ["ES1", "NQ1", "RTY1", "TY1", "CL1", "DXY", "UX1", "UX2"] if c in return_panel]
return_panel[sample_return_columns].dropna(subset=["ES1"]).head(5).round(6)


Returns are native one-bar log returns aligned by timestamp. Missing values are expected when one market has no matching bar on the ES clock.


## Page 2 - ES Impulse Definition

The event definition translates the research question into a point-in-time label: a large ES impulse is an hourly ES1 move whose absolute return exceeds the shifted rolling 95th percentile from the prior approximate 20-day ES window.


In [ ]:
events[["es_return_1h", "abs_es_return_1h", "rolling_abs_threshold", "large_abs", "large_down", "large_up"]].dropna().head(5).round(6)


The label table is the core point-in-time object. The threshold is shifted, so the current ES move cannot help define its own significance cutoff.


In [ ]:
event_summary.round(4)


The 95th percentile rule gives a usable but sparse event set. Down and up impulses are roughly balanced in this 140-day local window.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar(event_summary["event"], event_summary["event_count"], color=["#1f6f8b", "#9b3d2e", "#3a7f5f"])
axes[0].set_title("ES impulse event counts")
axes[0].set_ylabel("bars")
axes[1].bar(event_summary["event"], event_summary["mean_abs_es_return_bp"], color=["#1f6f8b", "#9b3d2e", "#3a7f5f"])
axes[1].set_title("Mean absolute ES move")
axes[1].set_ylabel("basis points")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_event_counts.png", dpi=160)
plt.show()


Large absolute impulses average roughly 60 basis points in this pull. This confirms the event rule is isolating meaningful intraday shocks rather than ordinary bars.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
plot_frame = events.dropna(subset=["es_return_1h"])
ax.plot(plot_frame.index, plot_frame["es_return_1h"] * 10_000, linewidth=0.75, color="#2f3a45", label="ES1 60m return")
ax.scatter(events.index[events["large_down"]], events.loc[events["large_down"], "es_return_1h"] * 10_000, s=18, color="#9b3d2e", label="large down")
ax.scatter(events.index[events["large_up"]], events.loc[events["large_up"], "es_return_1h"] * 10_000, s=18, color="#3a7f5f", label="large up")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("ES1 60-minute returns with impulse labels")
ax.set_ylabel("basis points")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_es_impulse_labels.png", dpi=160)
plt.show()


The plot is a visual audit of label placement. The labels attach to visibly large ES bars, which is the minimum sanity check before driver attribution.


## Page 3 - Conditional Correlation Benchmark

This is the baseline your boss asked for: when ES moves significantly, do simple cross-asset correlations identify the likely driver basket without PCA?


In [ ]:
for sample in ["threshold_ready", "large_abs", "large_down", "large_up"]:
    print(f"\n{sample}")
    display(
        corr.query("sample == @sample")
        .sort_values(["abs_corr_with_es", "n_obs"], ascending=[False, False])
        .head(10)
        .round(4)
    )


Conditional correlations already recover a coherent risk-off map. NQ and RTY move with ES, while VIX futures move strongly against ES during downside impulses.


In [ ]:
large_down_corr = (
    corr.query("sample == 'large_down'")
    .sort_values("abs_corr_with_es", ascending=True)
    .tail(12)
)
fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = np.where(large_down_corr["corr_with_es"] >= 0, "#1f6f8b", "#9b3d2e")
ax.barh(large_down_corr["driver"], large_down_corr["corr_with_es"], color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top conditional correlations during large ES downside moves")
ax.set_xlabel("correlation with ES1")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_large_down_correlations.png", dpi=160)
plt.show()


The chartbook view makes the risk-off basket legible quickly. Positive bars are equity/rates co-movement with ES; negative bars are volatility and commodity moves against ES.


In [ ]:
heatmap_drivers = (
    corr.query("sample in ['threshold_ready', 'large_abs', 'large_down', 'large_up']")
    .assign(rank=lambda x: x.groupby("sample")["abs_corr_with_es"].rank(ascending=False, method="first"))
    .query("rank <= 8")
    ["driver"]
    .drop_duplicates()
    .tolist()
)
heatmap = (
    corr[corr["driver"].isin(heatmap_drivers)]
    .pivot(index="driver", columns="sample", values="corr_with_es")
    .reindex(columns=["threshold_ready", "large_abs", "large_down", "large_up"])
)
fig, ax = plt.subplots(figsize=(8, 5.8))
im = ax.imshow(heatmap, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns, rotation=30, ha="right")
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set_title("Correlation map by ES impulse sample")
fig.colorbar(im, ax=ax, label="corr with ES1")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_correlation_heatmap.png", dpi=160)
plt.show()


The heatmap shows how driver relationships change across all, absolute, down, and up impulse samples. This is the right benchmark before claiming PCA adds new structure.


## Page 4 - Event-Conditioned PCA

PCA is fit on standardized driver returns, excluding ES1. Component signs are aligned so positive loadings correspond to positive ES co-movement.


In [ ]:
pca_summary.query("status == 'fit'").round(4)


Large absolute moves are more factor-concentrated than threshold-ready bars. This is the strongest reason to keep PCA in the workflow.


In [ ]:
pc1 = pca_summary.query("component == 1 and status == 'fit'").copy()
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(pc1["sample"], pc1["explained_variance_ratio"] * 100, color="#1f6f8b")
ax.set_title("PC1 variance share by conditioning sample")
ax.set_ylabel("percent of driver variance")
ax.set_ylim(0, max(75, pc1["explained_variance_ratio"].max() * 115))
for tick in ax.get_xticklabels():
    tick.set_rotation(25)
    tick.set_ha("right")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_pc1_variance.png", dpi=160)
plt.show()


PC1 explains about two-thirds of driver variance during large absolute ES moves. That is a clearer chartbook finding than the early predictive screen.


In [ ]:
for sample in ["large_abs", "large_down"]:
    loadings = (
        pca_loadings.query("sample == @sample and component == 1")
        .sort_values("abs_loading", ascending=True)
        .tail(12)
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = np.where(loadings["loading_aligned_to_es"] >= 0, "#1f6f8b", "#9b3d2e")
    ax.barh(loadings["driver"], loadings["loading_aligned_to_es"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"PC1 loadings aligned to ES: {sample}")
    ax.set_xlabel("loading")
    fig.tight_layout()
    fig.savefig(reports_dir / f"chartbook_pc1_loadings_{sample}.png", dpi=160)
    plt.show()


PC1 loadings translate the factor into a driver basket. The factor is not just equity beta; volatility, oil, rates, and FX loadings describe the cross-asset shape of the impulse.


In [ ]:
rolling_pca[["all_bar_absorption", "large_abs_absorption", "large_abs_rows"]].describe().round(4)


Rolling PCA concentration is consistently higher inside large ES move windows. This supports the idea that important impulses are more structured than the full intraday stream.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(rolling_pca["timestamp"], rolling_pca["all_bar_absorption"], label="all ES-clock bars", color="#1f6f8b")
ax.plot(rolling_pca["timestamp"], rolling_pca["large_abs_absorption"], label="large ES moves", color="#9b3d2e")
ax.set_title("Rolling PCA absorption: first three PCs")
ax.set_ylabel("variance share")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_rolling_absorption.png", dpi=160)
plt.show()


The visual compares background factor concentration with large-move concentration through time. Gaps occur when there are too few large events in the rolling window.


## Page 5 - Diagnostic Signal Screen

The signal screen asks whether known-at-bar-close variables help rank the next ES bar's impulse risk. This is intentionally diagnostic, not a production forecast.


In [ ]:
predictive_features.head(5).round(6)


These are the model inputs before fitting. Targets are next-ES-bar labels shifted along the ES clock, not the combined cross-asset timestamp grid.


In [ ]:
predictive.round(4)


Unweighted logistic improves ranking but barely improves Brier score. Balanced logistic is diagnostic only because its probabilities are not calibrated for rare-event quoting.


In [ ]:
metric_frame = predictive.query("model in ['train_base_rate', 'logit_unweighted']").copy()
metric_frame["label"] = metric_frame["target"].str.replace("target_next_", "", regex=False) + " / " + metric_frame["model"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(metric_frame["label"], metric_frame["average_precision"] * 100, color="#1f6f8b")
axes[0].set_title("Average precision")
axes[0].set_ylabel("percent")
axes[1].bar(metric_frame["label"], metric_frame["brier_score"], color="#7f8b99")
axes[1].set_title("Brier score")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(reports_dir / "chartbook_predictive_metrics.png", dpi=160)
plt.show()


The signal chart separates ranking from calibration. Average precision improves, but Brier gains are small, so the output should be framed as signal discovery.


In [ ]:
for target in coefficients["target"].drop_duplicates():
    top = (
        coefficients.query("target == @target and model == 'logit_unweighted'")
        .head(10)
        [["feature", "coefficient", "abs_coefficient"]]
        .round(4)
    )
    print(f"\n{target} / logit_unweighted")
    display(top)


Coefficients are a screening device, not causal evidence. They identify variables worth monitoring or testing with deeper history.


## Page 6 - Current Readout And Next Steps

The point of this page is to separate presentable findings from open research risk.


In [ ]:
large_abs_pc1 = pca_summary.query("sample == 'large_abs' and component == 1").iloc[0]
ready_pc1 = pca_summary.query("sample == 'threshold_ready' and component == 1").iloc[0]
absorption = rolling_pca[["all_bar_absorption", "large_abs_absorption"]].describe()
best_large_down_corr = corr.query("sample == 'large_down'").sort_values("abs_corr_with_es", ascending=False).head(5)
best_abs_model = predictive.query("target == 'target_next_large_abs' and model == 'logit_unweighted'").iloc[0]

print("Research readout")
print(f"- Large absolute ES impulses: PC1 explains {large_abs_pc1['explained_variance_ratio']:.1%} of driver variance versus {ready_pc1['explained_variance_ratio']:.1%} on threshold-ready bars.")
print(f"- Large-move PC1 is aligned with ES at abs corr {large_abs_pc1['abs_corr_with_es']:.2f}.")
print(f"- Median rolling absorption: all bars {absorption.loc['50%', 'all_bar_absorption']:.1%}, large ES moves {absorption.loc['50%', 'large_abs_absorption']:.1%}.")
print("- Top large-down pairwise drivers:")
for _, row in best_large_down_corr.iterrows():
    print(f"  {row['driver']}: corr {row['corr_with_es']:.2f} over {int(row['n_obs'])} observations")
print(f"- Next-bar large-absolute logistic screen: ROC AUC {best_abs_model['roc_auc']:.2f}, AP {best_abs_model['average_precision']:.2%}, Brier {best_abs_model['brier_score']:.4f}.")


The strongest current finding is attribution, not prediction. Large ES impulses compress into a more concentrated cross-asset factor; deeper history is needed for formal forecasting claims.
